# Exercise 12 - Finding outliers

In this exercise, you are to create a two-column data frame from the taxi data we looked at in exercise 6. The first column will contain the passenger count for each trip, and the second column will contain the distance (in miles) for each trip. Once you have created this data frame, I want you to:

* Count how many trip distances were outliers.
* Calculate the mean number of passengers for outliers. Is it different from the mean number of passengers for all trips?

1. Create data frame

In [31]:
import pandas as pd

# load Series from cvs files
taxi_passengers = pd.read_csv('../ch01/taxi-passenger-count.csv', header=None, names=['n_passengers', ]).squeeze()
ride_distance = pd.read_csv('../ch01/taxi-distance.csv', header=None, names=['distance_mi']).squeeze()

# create DataFrame from Series
taxi_rides = pd.DataFrame(
    {
        "n_passengers": taxi_passengers,
        "distance_mi": ride_distance,
    })

taxi_rides.iloc[:10] # preview of DataFrame

,n_passengers,distance_mi
0,1,1.63
1,1,0.46
2,1,0.87
3,1,2.13
4,1,1.40
5,1,1.40
6,1,1.80
7,4,11.90
8,1,1.27
9,1,0.60


* Count how many trip distances were outliers.

In [32]:
# 1. calculate interquantile range (IQR) for travel distance
iqr = taxi_rides["distance_mi"].quantile(0.75) - taxi_rides["distance_mi"].quantile(0.25)

# 2. calculate outliers thresholds
lower_lim = taxi_rides["distance_mi"].quantile(0.25) - 1.5 * iqr
upper_lim = taxi_rides["distance_mi"].quantile(0.75) + 1.5 * iqr

# count outliers
n_outliers = len(taxi_rides.query('(distance_mi < @lower_lim) | (distance_mi > @upper_lim)')) # note how to reference variables inside the query with @

print(f"iqr: {iqr}\nlower_lim: {lower_lim:.2f}\nupper_lim: {upper_lim}\nn_outliers: {n_outliers}")

iqr: 2.3
lower_lim: -2.45
upper_lim: 6.75
n_outliers: 1219


* Calculate the mean number of passengers for outliers. Is it different from the mean number of passengers for all trips?

In [33]:
mean_passengers_outliers = taxi_rides.query('(distance_mi < @lower_lim) | (distance_mi > @upper_lim)')["n_passengers"].mean()
mean_passengers = taxi_rides["n_passengers"].mean()

print(f"mean passengers for outliers: {mean_passengers_outliers:.2f}\nmean passengers for for all trips: {mean_passengers:.2f}")

mean passengers for outliers: 1.73
mean passengers for for all trips: 1.66


## Beyond the exercise

As I said earlier, there are several ways to define and find outliers. Let’s try a few different techniques:

* If you define outliers to be the lowest 10% and highest 10% of values, how many were there? Why is (or isn’t) this a good measure?

In [34]:
percentile_10 = taxi_rides["distance_mi"].quantile(0.10) # calculate 10% percentile
percentile_90 = taxi_rides["distance_mi"].quantile(0.9) # calculate 90% percentile

# calculate the number of outliers
n_outliers_10pct = len(taxi_rides.query("(distance_mi < @percentile_10) | (distance_mi > @percentile_90)"))

print(f"number of outliers: {n_outliers_10pct}")

number of outliers: 1984


This is a bad measure because it doesn't account for the spread of the data.

* How many short, medium, and long trips had only one passenger?


Assuming distances below 25% quantile are short, 25% quantile <= distances <= 75% quantile are medium, and distances > 75% quantile are long:

In [35]:
short_rides = taxi_rides["distance_mi"] < taxi_rides["distance_mi"].quantile(0.25) # boolean index for short rides
medium_rides = ((taxi_rides["distance_mi"] >= taxi_rides["distance_mi"].quantile(0.25)) & # boolean index for medium rides
               (taxi_rides["distance_mi"] <= taxi_rides["distance_mi"].quantile(0.75)))
long_rides = taxi_rides["distance_mi"] > taxi_rides["distance_mi"].quantile(0.75) # boolean index for long rides
one_passenger = taxi_rides["n_passengers"] == 1 # boolean index for one passenger rides

n_short = len(taxi_rides.loc[short_rides & one_passenger]) # calculate the number of short rides with one passenger
n_medium = len(taxi_rides.loc[medium_rides & one_passenger])
n_long = len(taxi_rides.loc[long_rides & one_passenger])

print(f"stats for one passenger rides:\nshort rides: {n_short}\n"
      f"medium rides: {n_medium}\nlong rides: {n_long}\n")

stats for one passenger rides:
short rides: 1812
medium rides: 3661
long rides: 1734



* The ```scipy.stats.zscore``` function rescales and centers (that is, normalizes) the data set. In this case, the mean is set to 0, and values can be above and below that value. Find all the distances for which the absolute value of the z-score is greater than 3.

In [36]:
from scipy.stats import zscore
from numpy import abs

normalized_distance = pd.Series(zscore(taxi_rides["distance_mi"]), name="distance_mi")

normalized_distance[:10]

0   -0.378596
1   -0.668393
2   -0.566840
3   -0.254751
4   -0.435565
5   -0.435565
6   -0.336489
7    2.165174
8   -0.467764
9   -0.633716
Name: distance_mi, dtype: float64

In [37]:
abs_distance_gt_3 = normalized_distance[abs(normalized_distance) > 3].count()

print(f"rides with an absolute distance greater than 3: {abs_distance_gt_3}")

rides with an absolute distance greater than 3: 306
